# Assignment 2
## P+7 (Oulipian language modelling)

### Background

The Oulipo (*Ouvroir de Littérature Potentielle*, or “Workshop of Potential Literature”) is a French literary group founded in 1960 by writer Raymond Queneau and mathematician François Le Lionnais. The group focuses on using rules and constraints in writing as a way to spark creativity. Rather than seeing constraints as obstacles, Oulipians treat them as tools to inspire new forms of storytelling and poetry. Their work combines mathematics, language, and playfulness, making their approach both unique and influential in modern literature.

One of the most famous Oulipian writers is Georges Perec, who is known for his creative use of constraints. His novel [*La Disparition* (“A Void”) is written entirely without the letter "e,"](https://archive.org/details/void00pere/mode/2up) which is especially challenging given how common "e" is in French. Perec’s writing often plays with the structure of language in surprising ways.



![George Perec](https://upload.wikimedia.org/wikipedia/commons/7/76/Myart_georges-perec_1978.jpg)

(George Perec, 1978. From Wikidata)

<!-- <img src="https://media.vigliensoni.com/clips/CART498/perec-01.jpg" width="800"> -->


One popular Oulipian technique is N+7, where each noun in a text is replaced by the noun seven entries later in a dictionary. This creates unusual, absurd, and often funny results, encouraging writers to think differently about language and meaning.

## The N+7 technique

One popular Oulipian technique is N+7, where each noun in a text is replaced by the noun seven entries later in a dictionary. This creates unusual, absurd, and often funny results, encouraging writers to think differently about language and meaning.

The N+7 technique process is straightforward

- **Start with a text**. Choose any text—this could be a poem, a sentence, or a passage.
- **Use a dictionary**. Have a dictionary (or word list) handy.
- **Replace each noun**. For every substantive noun in the original text, replace it with the noun appearing seven nouns away in the dictionary. If the end of the dictionary is reached, you can loop back to the beginning.
- **Maintain grammar**. Ensure the new text maintains grammatical correctness as much as possible, though the results often turn out  surreal and nonsensical.

For example, using the N+7 technique with a standard English dictionary for the original sentence:

*One must have a mind of winter*.

- ”mind” → 7 nouns after “mind” is “minimum” or “mindset” (depending on the dictionary)
- “winter” → 7 nouns after  “wishing” or “wipe”

Then, after applying the technique the results are:

- Original: “One must have a mind of winter”
- Dictionary one: “One must have a minimum of wishing”
- Dictionary two: “One must have a mindset of wipe”

[Online example](http://www.spoonbill.org/n+7/)

### Assignment and deliverables

For this assignment, you will create a variation of the N+7 technique we will name P+7. Using the GPT-2 language model, you will replace the last word of each line from *The Snow Man* with the token that has the seventh-highest probability according to the model’s predictions.

By the end of this assignment, submit a link to a GitHub repository named `CART498-GenAI` containing a folder labelled `A02` with the following items:

- A version of the text processed with your P+7 technique, saved as a `.txt` file.
- A second version of the text processed with `P+x`. Choose an `x` value that produces the funniest, wittiest, or most absurd version of the original text. Save this as a .txt file, and include the `x` value in the filename (e.g., `P+23.txt` or `P+12.txt`).
- A Python notebook with the script used to generate your P+7 and P+x transformations using the GPT-2 language model.
- A short reflection (250–350 words) explaining how altering the `x` value impacted the output of your P+x version. Additionally, discuss how you would implement a P+7 technique in which all nouns are replaced with their seventh-highest probability alternatives.

### *The Snow Man*
by Wallace Stevens (1879-1955)


> One must have a mind of winter  
> To regard the frost and the boughs  
> Of the pine-trees crusted with snow;  
> And have been cold a long time  
> To behold the junipers shagged with ice,  
> The spruces rough in the distant glitter  
> Of the January sun; and not to think  
> Of any misery in the sound of the wind,  
> In the sound of a few leaves,  
> Which is the sound of the land  
> Full of the same wind  
> That is blowing in the same bare place  
> For the listener, who listens in the snow,  
> And, nothing himself, beholds  
> Nothing that is not there and the nothing that is.  




In [5]:
import torch
import re
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# model & tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# Text (The Snow Man by Wallace Stevens)
poem_text = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

def is_valid_word(word):
    """Checks if a word is a valid, non-fragmented word, filtering out unnecessary special characters/spaces."""
    word_clean = word.strip()
    if not word_clean:
        return False
    # Exclude cases with only special characters, underscores (_), asterisks (*), etc. consecutively.
    # Allow hyphens and apostrophes within words, but block punctuation forms.
    # Allow words containing numbers (e.g., 'P+7', 'GPT-2').
    if not re.fullmatch(r"^[a-zA-Z0-9'-]*[a-zA-Z][a-zA-Z0-9'-]*$", word_clean):
        return False
    # Block very short consonant fragments or abnormal fragments (minimum 2 characters or only articles like 'a', 'I' allowed).
    if len(word_clean) == 1 and word_clean.lower() not in ['a', 'i', 'o']:
        return False
    return True

def process_poem_p_plus_x(text, x=7):
    lines = text.strip().split('\n')
    transformed_lines = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        tokens_in_line = line.split()
        last_word_raw = tokens_in_line[-1]

        # Separate punctuation from the last word
        match = re.match(r"^([a-zA-Z0-9'-]+)([\W_]*)$", last_word_raw)
        if match:
            last_word = match.group(1)
            punctuation = match.group(2)
        else:
            last_word = last_word_raw
            punctuation = ""

        # Generate context
        context_words = tokens_in_line[:-1]
        context_str = " ".join(context_words) + " " if context_words else ""

        inputs = tokenizer(context_str, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = outputs.logits[0, -1, :]

        # Sort top predictions
        sorted_probs, sorted_indices = torch.sort(predictions, descending=True)

        valid_candidates = []
        seen_words = set()

        # Filter for valid words
        for idx in sorted_indices.tolist():
            decoded_token = tokenizer.decode([idx])
            # Remove encoding special characters (like Ġ) and strip whitespace
            word_candidate = decoded_token.strip()

            if is_valid_word(word_candidate):
                # Prevent duplicate words (case-insensitive)
                word_lower = word_candidate.lower()
                if word_lower not in seen_words and word_lower != last_word.lower():
                    seen_words.add(word_lower)
                    valid_candidates.append(word_candidate)

            # Stop when enough candidates are gathered
            if len(valid_candidates) > max(50, x + 5):
                break

        # Select the x-th word from the refined candidates (using x as index for 0-indexed list)
        # If there are not enough valid candidates, use the last available candidate
        replacement_word = valid_candidates[min(x, len(valid_candidates) - 1)]

        new_line = f"{context_str}{replacement_word}{punctuation}"
        transformed_lines.append(new_line)

    return "\n".join(transformed_lines)

# Execute P+7 and save
p_7_result = process_poem_p_plus_x(poem_text, x=7)
print("=== P+7 Result ===")
print(p_7_result)

with open("P+7.txt", "w", encoding="utf-8") as f:
    f.write(p_7_result)

# Execute P+x and save
x_val = 40
p_x_result = process_poem_p_plus_x(poem_text, x=x_val)
print(f"\n=== P+{x_val} Result ===")
print(p_x_result)

with open(f"P+{x_val}.txt", "w", encoding="utf-8") as f:
    f.write(p_x_result)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

=== P+7 Result ===
One must have a mind of urn
To regard the frost and the urch
Of the pine-trees crusted with vern;
And have been cold a long ivan
To behold the junipers shagged with ery,
The spruces rough in the distant irc
Of the January sun; and not to ices
Of any misery in the sound of the iced,
In the sound of a few ich,
Which is the sound of the urs
Full of the same iz
That is blowing in the same bare urch
For the listener, who listens in the ips,
And, nothing himself, ices
Nothing that is not there and the nothing that ips.

=== P+40 Result ===
One must have a mind of idence
To regard the frost and the ural
Of the pine-trees crusted with izz;
And have been cold a long irit
To behold the junipers shagged with ides,
The spruces rough in the distant ingle
Of the January sun; and not to ily
Of any misery in the sound of the urd,
In the sound of a few ico,
Which is the sound of the izzy
Full of the same icks
That is blowing in the same bare iren
For the listener, who listens in the 